In [1]:
import os, torch
print("working folder:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

working folder: /content
GPU: Tesla T4


In [2]:
%cd /content
!git clone https://github.com/samyuktaathreya/pinyin_audio_recognizer_model.git repo
%cd /content/repo/pinyin_scorer
%pip install -q pypinyin

/content
fatal: destination path 'repo' already exists and is not an empty directory.
/content/repo/pinyin_scorer


In [3]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/pinyin_runs

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
stage1	stage2	stage3	stage4


In [4]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo log --oneline -1
!ls *.py
!grep -c "Hiss only" data.py

/content/repo/pinyin_scorer
877929f (HEAD -> main, origin/main, origin/HEAD) training a new model
data.py		 model.py	     prepare_folders.py    train.py
diagnose.py	 pinyin_units.py     prepare_syllables.py
eval_sources.py  prepare_aishell.py  score.py
1


In [3]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!unzip -q -o /content/drive/MyDrive/processed.zip -d /content/ext
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl
!python prepare_folders.py --root /content/ext/processed --out-dir manifests/folders

train: 6560 clips -> manifests/syl_train.jsonl
test: 3279 clips -> manifests/syl_test.jsonl
clips per speaker: {'FV1': 1640, 'FV2': 1640, 'FV3': 1639, 'MV1': 1640, 'MV2': 1640, 'MV3': 1640}
davinfifield     train   2485 clips
digmandarin      train   3254 clips
gitcode          test    1283 clips
hotoo            train   3232 clips
hugolpz          train   3312 clips  (8 skipped)
losttheory       train   3136 clips
mandarintemple   train   3206 clips
misc             train    336 clips
shikangkai_f     test    1615 clips
shikangkai_m     train   1425 clips
sreading         train   3232 clips
studycli         test    3144 clips
uvm              test    1300 clips
yabla            train   4884 clips
yoyochinese      train   4836 clips

train: 33338 clips (7.6 h)   test: 7342 clips


In [ ]:
!grep -c "Hiss only" data.py
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

1
Loading weights: 100% 211/211 [00:09<00:00, 21.43it/s]
39898 training clips, 8.9 hours, 626 batches per epoch, device=cuda, amp=True
before training initial_error=0.0360  final_error=0.0372  tone_error=0.0128  mean_error=0.0287  syllable_initial_acc=0.9643  syllable_final_acc=0.9637  syllable_tone_acc=0.9884  syllable_all3_acc=0.9283  syllable_clips=3279
/content/repo/pinyin_scorer/train.py:191: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
epoch 1 batch 50/626 step 50 initial=1.349 final=0.910 tone=0.280 lr=2.55e-06
epoch 1 batch 100/626 step 100 initial=1.062 final=0.866 tone=0.197 lr=5.05e-06
epoch 1 batch 150/626 step 150 i

In [5]:
!ps aux | grep "train.py" | grep -v grep | wc -l
!ls /content/repo/pinyin_scorer/manifests /content/ext 2>&1 | head -5
!ls /content/drive/MyDrive/pinyin_runs/stage4/last

3
/content/ext:
__MACOSX
processed

/content/repo/pinyin_scorer/manifests:
encoder  top.pt  units.json


In [6]:
%cd /content/repo/pinyin_scorer

/content/repo/pinyin_scorer


In [7]:
!setsid nohup python -u train.py --init-from /content/drive/MyDrive/pinyin_runs/stage4/last \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4b \
    --epochs 5 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 100 --augment --workers 2 \
    > /content/train_stage4b.log 2>&1 &

In [8]:
!tail -3 /content/train_stage4b.log
!grep "^epoch [0-9]* initial" /content/train_stage4b.log

In [9]:
!python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/stage4b/best \
    --manifests manifests/syl_test.jsonl manifests/folders/test_*.jsonl --out stage4_heldout.csv

Traceback (most recent call last):
  File "/content/repo/pinyin_scorer/eval_sources.py", line 19, in <module>
    from score import Scorer
  File "/content/repo/pinyin_scorer/score.py", line 22, in <module>
    from model import PinyinCTC
  File "/content/repo/pinyin_scorer/model.py", line 14, in <module>
    from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
  File "/usr/local/lib/python3.13/dist-packages/transformers/__init__.py", line 847, in <module>
    for _proc_file in sorted((Path(__file__).parent / "models").rglob("image_processing_*.py")):
                      ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/glob.py", line 505, in select_recursive
    yield from select_recursive_step(stack, match_pos)
  File "/usr/lib/python3.13/glob.py", line 529, in select_recursive_step
    yield from select_next(entry_path, exists=True)
  File "/usr/lib/python3.13/glob.py", line 456, in select_wildcard
    entries = list(

In [11]:
!ps aux | grep "train.py" | grep -v grep | grep -o "\-\-out [^ ]*" | sort -u

--out /content/drive/MyDrive/pinyin_runs/stage4


In [15]:
import json
for line in open("/content/drive/MyDrive/pinyin_runs/stage4/log.jsonl"):
    d = json.loads(line)
    print(d["epoch"], *(f"{d[k]:.3f}" for k in
          ["syllable_initial_acc", "syllable_final_acc", "syllable_tone_acc", "syllable_all3_acc"]))

1 0.972 0.894 0.942 0.842
2 0.967 0.937 0.939 0.866
3 0.950 0.865 0.892 0.780
4 0.969 0.914 0.935 0.852
5 0.969 0.864 0.846 0.755


In [16]:
!pkill -f train.py
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "no-augment-sources" prepare_folders.py; grep -c "duplicate_pairs" find_duplicates.py; grep -c "def speech_power" data.py

/content/repo/pinyin_scorer
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 37 (delta 4), reused 37 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (37/37), 461.90 KiB | 1.49 MiB/s, done.
From https://github.com/samyuktaathreya/pinyin_audio_recognizer_model
   877929f..a5c183c  main       -> origin/main
Updating 877929f..a5c183c
Fast-forward
 .gitignore                                         |   3 +
 .../digmandarin_lang1_002277_0_original.wav        | Bin 0 -> 25422 bytes
 augment_check/digmandarin_lang1_002277_1_noisy.wav | Bin 0 -> 25422 bytes
 augment_check/digmandarin_lang1_002277_2_noisy.wav | Bin 0 -> 39664 bytes
 augment_check/digmandarin_lang1_002277_3_noisy.wav | Bin 0 -> 38708 bytes
 .../digmandarin_si2_002823_0_original.wav          | Bin 0 -> 22566 bytes
 augment_check/digmandarin_si2_002823_1_noisy.wav   | Bin 0 -> 22566 bytes
 augment_check/digmandarin_si2_00282